# Recursive Descent Parser of Chemical Formulas
In this activity, you will implement a recursive descent parser to validate and interpret chemical formulas. This framework will combine several things we've been talking about, e.g., recursion, and a queue data structure to tell us if a chemical reaction expression is valid. 

A [recursive descent parser](https://en.wikipedia.org/wiki/Recursive_descent_parser) is a kind of top-down parser that recursively walks down a collection of objects until all the objects in the collection have been processed. Suppose we have comma delimited chemical reaction records of the form: `R00267,C6H8O7+C21H29N7O17P3,C5H6O5+CO2+C21H30N7O17P3+H` where:

* Field 1: The `name` field contains a unique string identifier for the reaction string, e.g., `R00267.`
* Field 2: The `reactants` field contains the reaction string, e.g., `C6H8O7+C21H29N7O17P3` where different reactants are separated by a `+` sign.
* Field 3: The `products` field contains the reaction string, e.g., `C5H6O5+CO2+C21H30N7O17P3+H` where the different chemical products are separated by a `+` sign.

Your recursive descent parser will be used to generate a stoichiometric matrix $\mathbf{S}\in\mathbb{R}^{s\times{r}}$. The stoichiometric matrix is a mathematical representation of the chemical reactions, where each row corresponds to a unique chemical species and each column corresponds to a reaction, i.e., it's the digital analog a set of chemical reaction equations. 

### Task Overview
* **Task 1**: Implement a recursive descent parser using queue data structures to split reaction strings into individual chemical species. This parser will process characters one by one to identify delimiter boundaries and extract species names.
* **Task 2**: Parse chemical reaction records to extract unique species and reaction names from the reaction data. We'll use Set data structures to automatically handle duplicates and ensure each species appears only once.
* **Task 3**: Construct a stoichiometric matrix that mathematically represents the chemical reaction network. Each matrix entry will encode whether a species is a reactant (-1), product (+1), or catalyst/intermediate (0) for each reaction.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [19]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Data
A coworker has provided us with a `CSV` file (`data/Reactions.net`) containing chemical reaction strings and some logic to parse these reactions in the `read_reaction_file(...)` function (defined below). The `Reactions.net` file contains records of the form:

```plaintext
# Sample biochemical reactions from Kegg TCA cycle: https://www.kegg.jp/pathway/map00020
# data records are comma separated where the fields are: name,reactants,products
R00267,C6H8O7+C21H29N7O17P3,C5H6O5+CO2+C21H30N7O17P3+H
R01082,C4H4O4+H2O,C4H6O5
R00342,C4H6O5+C21H28N7O14P2,C4H4O5+C21H29N7O14P2+H
R00352,C10H15N5O10P2+H3PO4+C23H38N7O17P3S+C4H4O5,C10H16N5O13P3+C6H8O7+C21H36N7O16P3S
R01324,C6H8O7,C6H8O7
R00405,C10H15N5O10P2+H3PO4+C25H40N7O19P3S,C10H16N5O13P3+C4H6O4+C21H36N7O16P3S
```

We need to read this file (which produces a collection of reaction records) and then parse each reaction string to generate a stoichiometric matrix.

### Implementation
We are provided with a function, `read_reaction_file(...)`, that reads the `Reactions.net` file and returns a dictionary of reaction records, where the key is reaction name, and the value is [a NamedTuple instance](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) with three fields: `name`, `reactants`, and `products`. The `reactants` and `products` fields contain the reaction strings that we need to parse.

__What is going on in this function?__

The `read_reaction_file(...)` function implements a simple CSV parser that processes chemical reaction data line by line. 
> The function uses Julia's safe file handling pattern `open(path, "r") do io ... end` which automatically closes the file when processing is complete, preventing resource leaks. 

As it iterates through each line using `eachline(io)`, the function filters out comment lines containing `#` characters using the `contains(line, "#")` function, ensuring only valid data records are processed. 
 > For each valid line, it splits the comma-separated values using `split(line, ',')` to extract the three fields: reaction name, reactants string, and products string. The function then constructs a NamedTuple with `(name=name, reactants=reactant, products=product)` and stores it in a dictionary using the reaction name as the key. 
 
 This approach provides memory-efficient processing since the file is read sequentially without loading everything into memory at once, and returns a well-structured `Dict{String, NamedTuple}` ready for further parsing operations.

In [ ]:
"""
    read_reaction_file(path::String) -> Dict{String, NamedTuple}

Reads a reaction file and returns a dictionary of reactions. Reaction information is stored as a `NamedTuple` with fields `name`, `reactants`, and `products`.

### Arguments
- `path::String`: The path to the reaction file.

### Returns
- `Dict{String, NamedTuple}`: A dictionary where the keys are reaction names and the values are `NamedTuple` instances containing the reaction details.
"""
function read_reaction_file(path::String)::Dict{String, NamedTuple}
    
    # TODO: Check: is path legit?
    # in production we would check this path, assume ok for now

    # initialize -
    reactions = Dict{String, NamedTuple}()

    # use example pattern from: https://varnerlab.github.io/CHEME-1800-Computing-Book/unit-1-basics/data-file-io.html#program-read-a-csv-file-refactored
    open(path, "r") do io # open a stream to the file
        for line in eachline(io) # read each line from the stream
            
            # Impl me -
            # line is a line from the file  

            # A couple of things to think about: 
            # a) ignore the comments, check out the contains function: https://docs.julialang.org/en/v1/base/strings/#Base.contains
            # b) records are comma delimited. Check out the split functions: https://docs.julialang.org/en/v1/base/strings/#Base.split
            # c) from the data in each record, we need to build a MyKeggReaction object. Each reaction object should be stored in the reactions dict with the name as the key
            if (contains(line,"#") == false)

                fields = split(line, ','); # splits around the ','
                
                # NOTE: For robustness, we should check if fields has at least 3 elements to handle malformed CSV lines,
                # but since we control the data file format, we'll assume the input is well-formed for this exercise.

                # grab the fields -
                name = string(fields[1]); # name of the reaction 
                reactant = string(fields[2]); # reactants
                product = string(fields[3]); # products

                # build NamedTuple and store -
                reactions[name] = (name=name, reactants=reactant, products=product);
            end
        end
    end

    # return -
    return reactions;
end;

Let's call the `read_reaction_file(...)` function to read the `Reactions.net` file and store the reaction records in a variable called `reactions::Dict{String, NamedTuple}`. This will give us a dictionary where the keys are the reaction names and the values are the corresponding NamedTuples containing the reactants and products.


In [21]:
reactions = let

    # initialize -
    path_to_reactions_file = joinpath(_PATH_TO_DATA, "Reactions.net");
    reactions = read_reaction_file(path_to_reactions_file);
end;

What's in each reaction record? We access the `reactions` dictionary using the reaction name as the key, e.g., `reactions["R00267"]` will give us a NamedTuple with the fields `name`, `reactants`, and `products`.

In [22]:
reactions["R00267"]

(name = "R00267", reactants = "C6H8O7+C21H29N7O17P3", products = "C5H6O5+CO2+C21H30N7O17P3+H")

 The `reactants` and `products` fields are strings that we need to parse to generate the stoichiometric matrix. Let's explore this next.

 ___

## Task 1: Recursive Descent Parser
In this task, you will implement a recursive descent parser to chop up the `reactants` and `products` strings into individual chemical species. 
We'll finish the implementation of the `recursivesplit(...)` function, which will take a string and a delimiter and return a vector of strings. This function will be used to split the `reactants` and `products` strings into individual chemical species.

However, before we do that with our reaction strings, let's consider a simple text string stored in the variable `test_input_string::String`:

In [23]:
test_input_string = "Dog+Cat+Mouse+Lizard"; # let's play with the recursivesplit function using 

We should be able to hand this string and the delimiter `+` to the `recursivesplit(...)` function and get back a vector of strings, e.g., `["Dog", "Cat", "Mouse", "Lizard"]` (in that order).

Let's implement the `recursivesplit(...)` function to do this. The function will take a string and a delimiter and return a vector of strings.

__What is going on in this function?__

The `recursivesplit(...)` function implements a recursive descent parser using queue data structures to methodically process strings character by character. The main function serves as a wrapper that converts the input string into a character array and populates a primary queue `q` with these characters. 
 - __Recursive case:__ It then calls the core recursive function `_recursive_descent_parser!(...)` which processes the queue until empty. The recursive parser operates with two queues: the main queue `q` containing unprocessed characters and a temporary queue `tmp` that accumulates characters to build words. At each recursive call, the function dequeues a character from the main queue and checks if it matches the delimiter. If a delimiter is found, it converts the accumulated characters in the temporary queue into a word using `join(tmp)`, adds it to the results array, and clears the temporary queue. If the character is not a delimiter, it gets added to the temporary queue for word building. 
 
The base case occurs when the main queue is empty, at which point any remaining characters in the temporary queue are converted to a final word and added to the results. This approach demonstrates how recursion can elegantly handle string parsing without explicit loop constructs, building the result incrementally through recursive calls.

In [ ]:
"""
    _recursive_descent_parser(q::Queue, tmp::Queue{Char}, a::Array{String,1}; 
        delim = ' ') -> Nothing

The `_recursive_descent_parser` function is a recursive descent parser that splits a string into an array of words.
The function uses a queue data structure to store the characters of the input string. 

### Arguments
- `q::Queue`: a queue of characters.
- `tmp::Queue{Char}`: a temporary queue of characters.
- `a::Array{String,1}`: an array of strings.
- `delim::Char=' '`: the delimiter to split the string on.

### Returns
- `Nothing`: the function returns nothing. It updates the array `a` in place.
"""
function _recursive_descent_parser!(q::Queue, tmp::Queue{Char}, a::Array{String,1}; 
    delim = ' ')::Nothing
    
    
    if (isempty(q) == true)
        if (isempty(tmp) == false)
            word = join(tmp)
            if (isempty(word) == false)
                push!(a,word)
            end
        end
        return nothing 
    else
        next_char = dequeue!(q)
        if (next_char == delim)
            word = join(tmp)
            if (isempty(word) == false)
                push!(a, word)
            end
            empty!(tmp);
        else
            enqueue!(tmp, next_char)
        end
        _recursive_descent_parser!(q, tmp, a; delim = delim);
    end
end

"""
    recursivesplit(string::String; delim::Char=' ') -> Array{String,1}

The `recursivesplit` function is a recursive descent parser that splits a string into an array of words. 
The function is a wrapper around the `_recursive_descent_parser!` function. 


### Arguments
- `string::String`: the string to split.
- `delim::Char=' '`: the delimiter to split the string on.

### Returns
- `Array{String,1}`: an array of words.
"""
function recursivesplit(string::String; delim::Char=' ')::Array{String,1}

    # initialize -
    tmp = Queue{Char}(); # temporary queue to hold characters while building words
    q = Queue{Char}(); # queue to hold characters from the string (we process this until it is empty)
    words = Array{String,1}(); # array to hold the words we build from the string


    character_arr = string |> collect # turn the string into an array of characters
    for c ∈ character_arr
        enqueue!(q, c); # add the characters to the q queue
    end

    # Fill me in here.
    _recursive_descent_parser!(q, tmp, words; delim = delim); # call the recursive descent parser
    
    # return words to the caller -
    return words;
end;

If our recursive descent parser is correct, we should return a vector of strings containing the animal names in the order they appear in the input string.  Let's check that it is correct [using the `@test` macro exported by `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/).

In [35]:
let
    # initialize -
    expected_output_vector = ["Dog", "Cat", "Mouse", "Lizard"]; # expected output
    our_output_vector = recursivesplit(test_input_string; delim = '+'); # call the recursivesplit function

    # check: if NOT equal ... then BOOOOM!
    @test our_output_vector == expected_output_vector; # check that the output is as expected
end

Test Passed

Ok, take a breath! The `recursivesplit(...)` function is a bit tricky, but it is a good exercise in recursion and parsing. However, it is not the end of the world if you don't get it right on the first try. The important thing is to understand the logic behind the recursive descent parser and how it works.

Now that we have the `recursivesplit(...)` function working, we can use it to parse the `reactants` and `products` strings in the reaction records.

## Task 2: Parsing Reaction Records
In this task, you will use the `recursivesplit(...)` function to parse the `reactants` and `products` strings in the reaction records to generate the __set__ of unique chemical species, and the set of unique reaction names.

> __Why a Set{String}?__ A set is a collection of __unique elements__, which means that it automatically handles duplicates for us. This is important because we want to ensure that each chemical species is only represented once in our final list. No need to check if an element is already in the collection before adding it; the set takes care of that. Cool!

Let's store the set of unique chemical species which in the `species::Set{String}` variable.

In [43]:
species = let
    
    # initialize -
    species = Set{String}(); # set to hold unique chemical species
    delim = '+'; # delimiter for the recursivesplit function

    for (key, reaction) ∈ reactions # iterate over the reactions dictionary
        tmp = recursivesplit(reaction.reactants; delim = delim) ∪ recursivesplit(reaction.products; delim = delim); # wow! fancy

        # add to the set of species -
        for s ∈ tmp
            push!(species, s);
        end
    end

    species; # return the set of species
end;

There's a lot going on in that species parsing code! Let's break down the key operations. 

The code iterates through each reaction in our dictionary and uses our recursive descent parser to extract both reactants and products. 
> __Union?__ The union operation (`∪`) combines the reactants and products for each reaction into a single collection, and then we add each species to our set. This approach ensures we capture all species that participate in any capacity across all reactions.

Next, let's get the list of reaction names. This is straightforward since the reaction names are already unique in the `reactions` dictionary. We can simply extract the keys from the dictionary and store them in a vector.

In [47]:
reaction_names = keys(reactions) |> collect |> sort; # get the reaction names as a vector

Now that we have the unique chemical species and reaction names, we can proceed to construct the stoichiometric matrix $\mathbf{S}\in\mathbb{R}^{s\times{r}}$!

## Task 3: Constructing the Stoichiometric Matrix
In this task, you will construct the stoichiometric matrix $\mathbf{S}\in\mathbb{R}^{s\times{r}}$ using the unique chemical species and reaction names.

The stoichiometric matrix construction process systematically encodes the participation of each chemical species in each reaction using a structured mathematical representation. The code begins by determining the matrix dimensions: `s` represents the number of unique species (rows) and `r` represents the number of reactions (columns), creating an integer matrix initialized with zeros. 
> __Order?__ To enable consistent indexing, the species set is converted to a sorted vector using `species |> collect |> sort`, ensuring reproducible row ordering. In a similar way, we've sorted the reaction names to maintain a consistent column order in the matrix. If we did not sort the species and reactions, the order could change each time we run the code, leading to inconsistencies in the stoichiometric matrix.

The algorithm iterates through each reaction, using our recursive descent parser to extract the individual reactants and products for that specific reaction. For each species-reaction pair, the code applies a logical classification scheme: 
> __Coefficients?__ if a species appears only as a reactant, it receives a coefficient of -1; if it appears only as a product, it gets +1; if it appears as both reactant and product (indicating a catalyst or intermediate), it gets 0; and if it doesn't participate at all, it remains 0. This classification works in this case because we are not dealing with reactions like: `2A -> B + C`, where the stoichiometric coefficients are not 1 or -1.

This systematic approach transforms the textual chemical reaction representations into a numerical matrix where each entry $S_{ij}$ represents the stoichiometric coefficient of species $i$ in reaction $j$, providing a mathematical foundation for further analysis such as reaction network analysis, flux balance analysis, or chemical equilibrium calculations (which we will explore in future activities).

Let's save the stoichiometric matrix in the `S::Matrix{Int}` variable.

In [49]:
S = let

    # initialize -
    s = length(species); # number of unique species
    r = length(reaction_names); # number of reactions
    S = zeros(Int, s, r); # stoichiometric matrix as Integer matrix
    delim = '+'; # delimiter for the recursivesplit function
    species_vector = species |> collect |> sort; # convert the set of species to a vector for indexing

    for i ∈ eachindex(reaction_names)

        # get the reaction model -
        reaction = reactions[reaction_names[i]]; # get the reaction record
        reactants = recursivesplit(reaction.reactants; delim = delim); # parse the reactants for this reaction
        products = recursivesplit(reaction.products; delim = delim); # parse the products for this reaction

        # process each species, given the reactants and products for this reaction
        for k ∈ eachindex(species_vector)
            
            species_symbol = species_vector[k]; # get the species symbol
            if (species_symbol ∈ reactants) && (species_symbol ∉ products)
                S[k,i] = -1; # reactant
            elseif (species_symbol ∉ reactants) && (species_symbol ∈ products)
                S[k,i] = 1; # product
            elseif (species_symbol ∈ reactants) && (species_symbol ∈ products)
                S[k,i] = 0; # both reactant and product
            end
        end
    end

    S; # return the stoichiometric matrix
end

19×6 Matrix{Int64}:
  0   0  -1  -1   0  0
  0   0   1   1   0  0
  0  -1   0   0   0  0
  0   1   0   0   0  0
 -1   0   0   0   0  0
  1   0   0   0   0  0
  0   0   1   1   0  0
  0   0  -1   0   0  0
  0   0   0  -1   0  0
  0   0   0   0  -1  0
  0   1  -1   0   0  0
  0   0   0   1   0  0
  0  -1   0   0   1  0
  1   0   0   0   0  0
 -1   0   1   0   0  0
  1   0   0   0   0  0
  1   1   0   0   0  0
  0   0   0   0  -1  0
  0   0  -1  -1   0  0

## Summary
In this activity, we successfully implemented a recursive descent parser to process chemical reaction expressions and generate stoichiometric matrices from textual reaction data. We began by developing a CSV parser that reads chemical reaction records from a file, extracting reaction names, reactants, and products into structured NamedTuple objects for systematic processing.

The core of our implementation was the recursive descent parser using queue data structures, which methodically splits reaction strings into individual chemical species by processing characters one at a time until delimiter boundaries are found. This parser enabled us to extract unique chemical species from both reactants and products across all reactions, automatically handling duplicates through Julia's Set data structure.

Finally, we constructed a stoichiometric matrix that mathematically represents the chemical reaction network, where each entry encodes whether a species is a reactant (-1), product (+1), or catalyst/intermediate (0) for each reaction. This matrix provides a foundation for advanced chemical engineering analyses such as reaction network studies, flux balance analysis, and chemical equilibrium calculations.